# 04 - Ingestão Bronze: Micro-Lotes em Delta Lake com Metadados
**Squad 2 — Real Time for Business | Dupla 1**  
**Integrantes:** Lucas Sousa Santos Oliveira & Zaiden Emiliano Segundo Seleme  
**Tabelas de Escopo:** `ecommerce_produtos` e `ecommerce_categorias`  
**Branch:** `feat/squad2-lucas_zaiden`  

### Objetivo da Task (Sprint 2 - Task 1):
1. **Detecção Incremental Nativa no Spark:** Identificar novos arquivos `.parquet` depositados no bucket `raw/real-time-data/` via leitura paralela distribuída e anti-join contra a tabela Delta de controle de ingestão.
2. **Auditoria e Rastreabilidade:** Adicionar em cada registro as colunas obrigatórias:
   * `bronze_ingested_at`: timestamp do momento da ingestão via `current_timestamp()`.
   * `bronze_source_file`: caminho completo do arquivo de origem via `col("_metadata.file_path")`.
3. **Zero Transformação (Regra de Ouro Bronze):** Não aplicar limpeza, filtros de regras de negócio ou alteração de tipos. O dado bruto é preservado 100% íntegro.
4. **Particionamento Temporal Obrigatório:** Salvar em formato **Delta Lake** em modo **`append`**, particionado por data de ingestão (`ano`, `mes`, `dia`, `hora`).
5. **Isolamento de Namespace (`/grupo1/`):** Gravar os dados no container `squad2` sob o prefixo seguro `/grupo1/` para evitar conflito com outras duplas.
6. **Controle de Idempotência e Metadados:** Registrar os arquivos processados na tabela Delta de controle `squad2.ingestion_control_log`.

## 1. Carregamento Seguro das Credenciais e Variáveis de Ambiente

In [0]:
import os
from dotenv import load_dotenv, find_dotenv

# Carregamento automático do arquivo .env com override
dotenv_path = find_dotenv()
if not dotenv_path:
    candidatos = [
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.getcwd()), ".env"),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env")
    ]
    for c in candidatos:
        if os.path.exists(c):
            dotenv_path = c
            break

load_dotenv(dotenv_path, override=True)

storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

print("Ambiente configurado com sucesso:")
print(f"  Storage Account: {storage_account}")
print(f"  Client ID disponível: {client_id is not None}")
print(f"  Tenant ID disponível: {tenant_id is not None}")
print(f"  Client Secret disponível: {client_secret is not None}")

## 2. Configurações de Conexão OAuth e Definição dos Caminhos ABFSS
Injetamos as credenciais via `adls_options` diretamente nos leitores e escritores do Spark, compatível 100% com Databricks Serverless sem dependência de bibliotecas externas adicionais.

In [0]:
# Configurações OAuth do Service Principal para injeção granular nas operações do cluster
adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# Definição dos caminhos com isolamento de namespace no prefixo /grupo1/
base_raw = f"abfss://raw@{storage_account}.dfs.core.windows.net/real-time-data"
base_squad = f"abfss://squad2@{storage_account}.dfs.core.windows.net/grupo1"

caminhos = {
    "origem_produtos": f"{base_raw}/*/*/*/*/ecommerce_produtos.parquet",
    "origem_categorias": f"{base_raw}/*/*/*/*/ecommerce_categorias.parquet",
    "bronze_produtos": f"{base_squad}/bronze/ecommerce_produtos",
    "bronze_categorias": f"{base_squad}/bronze/ecommerce_categorias",
    "metadata_control_log": f"{base_squad}/metadata/ingestion_control_log"
}

print("Caminhos configurados no Data Lake:")
for k, v in caminhos.items():
    print(f"  {k}: {v}")

## 3. Ingestão Incremental Bronze: `ecommerce_produtos`
Realiza a leitura distribuída via Spark, identifica arquivos inéditos comparando contra a tabela de controle Delta, adiciona os metadados de auditoria e grava na camada Bronze particionada por `ano`, `mes`, `dia`, `hora`.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql.functions import current_timestamp, year, month, dayofmonth, hour, col, count

print(f"Varrendo arquivos de produtos em: {caminhos['origem_produtos']}")

# 1. Leitura de todos os arquivos de produtos com a coluna oculta de caminho nativa do Spark
df_raw_prod = (spark.read
    .options(**adls_options)
    .parquet(caminhos["origem_produtos"])
    .withColumn("bronze_source_file", col("_metadata.file_path")))

# 2. Filtrar apenas arquivos que ainda não foram processados (Left Anti Join com a tabela de controle)
try:
    df_control_prod = spark.read.format("delta").options(**adls_options).load(caminhos["metadata_control_log"])
    df_novos_prod = df_raw_prod.join(
        df_control_prod.filter(col("tabela_origem") == "ecommerce_produtos"),
        df_raw_prod.bronze_source_file == df_control_prod.arquivo_processado,
        "left_anti"
    )
except Exception:
    # Primeira execução (tabela de controle ainda não existe)
    df_novos_prod = df_raw_prod

total_novos_prod = df_novos_prod.count()

if total_novos_prod == 0:
    print("Nenhum arquivo novo detectado para 'ecommerce_produtos'. Camada Bronze está atualizada!")
else:
    # Identificar resumo dos arquivos detectados
    arquivos_novos_prod = (df_novos_prod
        .groupBy("bronze_source_file")
        .agg(count("*").alias("linhas"))
        .collect())
    
    print(f"\n>>> Detectados {len(arquivos_novos_prod)} novos arquivos ({total_novos_prod} linhas totais):")
    for r in arquivos_novos_prod:
        print(f"    -> {r['bronze_source_file']} ({r['linhas']} linhas)")

    # 3. Adicionar metadados de auditoria e colunas temporais de partição (ZERO transformação no dado bruto)
    df_bronze_prod = (df_novos_prod
        .withColumn("bronze_ingested_at", current_timestamp())
        .withColumn("ano", year(col("bronze_ingested_at")))
        .withColumn("mes", month(col("bronze_ingested_at")))
        .withColumn("dia", dayofmonth(col("bronze_ingested_at")))
        .withColumn("hora", hour(col("bronze_ingested_at"))))

    # 4. Gravação Delta Bronze em modo append particionado
    (df_bronze_prod.write
        .format("delta")
        .options(**adls_options)
        .mode("append")
        .partitionBy("ano", "mes", "dia", "hora")
        .save(caminhos["bronze_produtos"]))
    
    # 5. Registrar cada arquivo processado na tabela de controle de auditoria
    schema_log = StructType([
        StructField("tabela_origem", StringType(), False),
        StructField("arquivo_processado", StringType(), False),
        StructField("total_linhas", LongType(), False),
        StructField("status", StringType(), False)
    ])
    
    log_data_prod = [("ecommerce_produtos", r["bronze_source_file"], int(r["linhas"]), "SUCCESS") for r in arquivos_novos_prod]
    df_log_prod = (spark.createDataFrame(log_data_prod, schema=schema_log)
                   .withColumn("timestamp_processamento", current_timestamp()))
    
    (df_log_prod.write
        .format("delta")
        .options(**adls_options)
        .mode("append")
        .save(caminhos["metadata_control_log"]))
    
    print(f"<<< Ingestão de 'ecommerce_produtos' finalizada com sucesso!")

## 4. Ingestão Incremental Bronze: `ecommerce_categorias`
Executa o mesmo fluxo distribuído para processar os arquivos de categorias com auditoria e particionamento temporal.

In [0]:
print(f"Varrendo arquivos de categorias em: {caminhos['origem_categorias']}")

# 1. Leitura de todos os arquivos de categorias com a coluna oculta de caminho nativa do Spark
df_raw_cat = (spark.read
    .options(**adls_options)
    .parquet(caminhos["origem_categorias"])
    .withColumn("bronze_source_file", col("_metadata.file_path")))

# 2. Filtrar apenas arquivos que ainda não foram processados (Left Anti Join com a tabela de controle)
try:
    df_control_cat = spark.read.format("delta").options(**adls_options).load(caminhos["metadata_control_log"])
    df_novos_cat = df_raw_cat.join(
        df_control_cat.filter(col("tabela_origem") == "ecommerce_categorias"),
        df_raw_cat.bronze_source_file == df_control_cat.arquivo_processado,
        "left_anti"
    )
except Exception:
    df_novos_cat = df_raw_cat

total_novos_cat = df_novos_cat.count()

if total_novos_cat == 0:
    print("Nenhum arquivo novo detectado para 'ecommerce_categorias'. Camada Bronze está atualizada!")
else:
    arquivos_novos_cat = (df_novos_cat
        .groupBy("bronze_source_file")
        .agg(count("*").alias("linhas"))
        .collect())
    
    print(f"\n>>> Detectados {len(arquivos_novos_cat)} novos arquivos ({total_novos_cat} linhas totais):")
    for r in arquivos_novos_cat:
        print(f"    -> {r['bronze_source_file']} ({r['linhas']} linhas)")

    # 3. Adicionar metadados de auditoria e colunas temporais de partição (ZERO transformação no dado bruto)
    df_bronze_cat = (df_novos_cat
        .withColumn("bronze_ingested_at", current_timestamp())
        .withColumn("ano", year(col("bronze_ingested_at")))
        .withColumn("mes", month(col("bronze_ingested_at")))
        .withColumn("dia", dayofmonth(col("bronze_ingested_at")))
        .withColumn("hora", hour(col("bronze_ingested_at"))))

    # 4. Gravação Delta Bronze em modo append particionado
    (df_bronze_cat.write
        .format("delta")
        .options(**adls_options)
        .mode("append")
        .partitionBy("ano", "mes", "dia", "hora")
        .save(caminhos["bronze_categorias"]))
    
    # 5. Registrar cada arquivo processado na tabela de controle de auditoria
    log_data_cat = [("ecommerce_categorias", r["bronze_source_file"], int(r["linhas"]), "SUCCESS") for r in arquivos_novos_cat]
    df_log_cat = (spark.createDataFrame(log_data_cat, schema=schema_log)
                  .withColumn("timestamp_processamento", current_timestamp()))
    
    (df_log_cat.write
        .format("delta")
        .options(**adls_options)
        .mode("append")
        .save(caminhos["metadata_control_log"]))
    
    print(f"<<< Ingestão de 'ecommerce_categorias' finalizada com sucesso!")

## 5. Criação e Mapeamento das Tabelas Externas no Databricks Metastore
Mapeia as tabelas Bronze sob o schema `squad2` apontando para o namespace seguro `/grupo1/` no Data Lake.

In [0]:
try:
    spark.sql("CREATE SCHEMA IF NOT EXISTS squad2")

    # Registrar tabela externa de produtos
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS squad2.bronze_ecommerce_produtos
    USING DELTA
    LOCATION '{caminhos["bronze_produtos"]}'
    """)

    # Registrar tabela externa de categorias
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS squad2.bronze_ecommerce_categorias
    USING DELTA
    LOCATION '{caminhos["bronze_categorias"]}'
    """)

    # Registrar tabela externa de controle de ingestão
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS squad2.ingestion_control_log
    USING DELTA
    LOCATION '{caminhos["metadata_control_log"]}'
    """)
    print("Tabelas externas registradas com sucesso no schema squad2!")
except Exception as e:
    print(f"Nota sobre Metastore Externo: {e}")
    print("Os dados Delta no ADLS e views temporárias estão prontos para consulta local.")

## 6. Auditoria Pós-Carga e Validação das Partições Físicas

In [0]:
print("=== Auditoria da Camada Bronze ===")

# 1. Leitura direta via Delta no Data Lake com autenticação granular adls_options
df_check_prod = spark.read.format("delta").options(**adls_options).load(caminhos["bronze_produtos"])
df_check_cat = spark.read.format("delta").options(**adls_options).load(caminhos["bronze_categorias"])
df_check_log = spark.read.format("delta").options(**adls_options).load(caminhos["metadata_control_log"])

# Criar views temporárias para consultas SQL diretas
df_check_prod.createOrReplaceTempView("bronze_ecommerce_produtos")
df_check_cat.createOrReplaceTempView("bronze_ecommerce_categorias")
df_check_log.createOrReplaceTempView("ingestion_control_log")

count_prod = df_check_prod.count()
count_cat = df_check_cat.count()
count_log = df_check_log.count()

print(f"Total de registros em bronze_ecommerce_produtos:   {count_prod}")
print(f"Total de registros em bronze_ecommerce_categorias: {count_cat}")
print(f"Total de registros na tabela de controle de logs:   {count_log}")

# 2. Verificação das partições temporais criadas
print("\n--- Partições Temporais Detectadas em Produtos ---")
display(df_check_prod.groupBy("ano", "mes", "dia", "hora").count().orderBy("ano", "mes", "dia", "hora"))

# 3. Exibir amostra dos metadados de auditoria
print("\n--- Amostra de Registros da Camada Bronze (Produtos) ---")
display(df_check_prod.select("sku", "nome_produto", "preco_lista", "bronze_ingested_at", "bronze_source_file").limit(5))

# 4. Exibir o histórico da tabela de controle de arquivos
print("\n--- Histórico de Ingestão (ingestion_control_log) ---")
display(df_check_log.orderBy(col("timestamp_processamento").desc()))